# imports and data loading

In [41]:
%env DATA_DIR=/kaggle/input/datasets/stutigandhi123/overtone-dataaa

env: DATA_DIR=/kaggle/input/datasets/stutigandhi123/overtone-dataaa


In [42]:
import json, os
import numpy as np
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold



def load_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows
DATA_DIR = os.environ.get("DATA_DIR", "./data")

with open(os.path.join(DATA_DIR, "labels.txt")) as f:
    labels = [ln.strip() for ln in f if ln.strip()]

train = load_jsonl(os.path.join(DATA_DIR, "train.jsonl"))
val   = load_jsonl(os.path.join(DATA_DIR, "val.jsonl"))
test  = load_jsonl(os.path.join(DATA_DIR, "test.jsonl"))
print(f"train {len(train)}, val {len(val)}, test {len(test)}, labels {len(labels)}")

train 34000, val 6000, test 13263, labels 27


# features: word + char n-grams, fit on train only

In [43]:
train_texts = [r["text"] for r in train]
val_texts   = [r["text"] for r in val]
test_texts  = [r["text"] for r in test]

# min_df=1 (row 4) recovers rare-label-specific phrases min_df=2 would drop.
# word(1,2) + char_wb(3,5) union (row 5) — char n-grams catch subword overlap
# that pure word n-grams miss (e.g. "bereaved" vs "bereavement").
word_vec = TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=1,
                            max_features=100000, sublinear_tf=True)
char_vec = TfidfVectorizer(lowercase=True, analyzer="char_wb", ngram_range=(3, 5),
                            min_df=1, max_features=100000, sublinear_tf=True)

# row 1: fit on train text ONLY -- no test leakage
word_vec.fit(train_texts)
char_vec.fit(train_texts)

def featurize(texts):
    return hstack([word_vec.transform(texts), char_vec.transform(texts)]).tocsr()

Xtr = featurize(train_texts)
Xva = featurize(val_texts)
Xte = featurize(test_texts)

print(f"word vocab {len(word_vec.vocabulary_)}, char vocab {len(char_vec.vocabulary_)}")

word vocab 100000, char vocab 100000


# train one classifier per label, on ALL rows

In [44]:
# use every row in `train` (not just single-label ones), so each
# classifier sees multi-label positives and zero-label negatives too.
clfs = {}
for lab in labels:
    y = np.array([1 if lab in r["labels"] else 0 for r in train])
    if y.sum() < 2:
        clfs[lab] = None
        continue
    clf = LogisticRegression(max_iter=1000, C=4.0, class_weight="balanced")
    clf.fit(Xtr, y)
    clfs[lab] = clf
    print(f"  {lab:<16} fitted on {int(y.sum())} positives")

def score_matrix(X):
    scores = np.zeros((X.shape[0], len(labels)))
    for j, lab in enumerate(labels):
        if clfs[lab] is not None:
            scores[:, j] = clfs[lab].predict_proba(X)[:, 1]
    return scores

val_scores  = score_matrix(Xva)
test_scores = score_matrix(Xte)

  admiration       fitted on 3166 positives
  amusement        fitted on 1832 positives
  anger            fitted on 1237 positives
  annoyance        fitted on 1926 positives
  approval         fitted on 2319 positives
  caring           fitted on 857 positives
  confusion        fitted on 1040 positives
  curiosity        fitted on 1715 positives
  desire           fitted on 472 positives
  disappointment   fitted on 997 positives
  disapproval      fitted on 1639 positives
  disgust          fitted on 649 positives
  embarrassment    fitted on 235 positives
  excitement       fitted on 635 positives
  fear             fitted on 493 positives
  gratitude        fitted on 2051 positives
  grief            fitted on 65 positives
  joy              fitted on 1132 positives
  love             fitted on 1658 positives
  nervousness      fitted on 131 positives
  optimism         fitted on 1257 positives
  pride            fitted on 95 positives
  realization      fitted on 860 positives
 

# helper functions: macro-F1 and threshold tuning

In [45]:
def macro_f1(pred_sets, gold_sets, labels):
    total = 0.0
    for lab in labels:
        tp = fp = fn = 0
        for p, g in zip(pred_sets, gold_sets):
            if lab in p and lab in g: tp += 1
            elif lab in p: fp += 1
            elif lab in g: fn += 1
        prec = tp/(tp+fp) if (tp+fp) else 0.0
        rec  = tp/(tp+fn) if (tp+fn) else 0.0
        total += 2*prec*rec/(prec+rec) if (prec+rec) else 0.0
    return total / len(labels)

def tune_thresholds(scores, gold_sets, labels):
    grid = np.arange(0.05, 0.96, 0.01)
    thresholds = np.full(len(labels), 0.5)
    for j, lab in enumerate(labels):
        gold = np.array([1 if lab in g else 0 for g in gold_sets])
        best_t, best_f1 = 0.5, -1.0
        for t in grid:
            pred = (scores[:, j] >= t).astype(int)
            tp = int((pred & gold).sum()); fp = int((pred & (1-gold)).sum()); fn = int(((1-pred) & gold).sum())
            prec = tp/(tp+fp) if (tp+fp) else 0.0
            rec  = tp/(tp+fn) if (tp+fn) else 0.0
            f1 = 2*prec*rec/(prec+rec) if (prec+rec) else 0.0
            if f1 > best_f1: best_t, best_f1 = float(t), f1
        thresholds[j] = best_t
    return thresholds

def decode(scores, thresholds, labels):
    out = []
    for i in range(scores.shape[0]):
        out.append({labels[j] for j in range(len(labels)) if scores[i,j] >= thresholds[j]})
    return out

val_gold = [set(r["labels"]) for r in val]

# honest k-fold evaluation (report this number, don't use it for shipping)

In [46]:
K = 5
kf = KFold(n_splits=K, shuffle=True, random_state=0)
fold_scores = []
for tune_idx, report_idx in kf.split(val_scores):
    th = tune_thresholds(val_scores[tune_idx], [val_gold[i] for i in tune_idx], labels)
    pred = decode(val_scores[report_idx], th, labels)
    gold = [val_gold[i] for i in report_idx]
    fold_scores.append(macro_f1(pred, gold, labels))

honest_mean = np.mean(fold_scores)
honest_std  = np.std(fold_scores)
print(f"honest k-fold macro-F1: {honest_mean:.4f} +/- {honest_std:.4f}")

honest k-fold macro-F1: 0.4378 +/- 0.0138


# final shipping thresholds + write predictions.jsonl

In [47]:
# Tune thresholds on the FULL val set for shipping -- this is fine, since
# test.jsonl is never touched during tuning, so no leakage against test.
ship_thresholds = tune_thresholds(val_scores, val_gold, labels)

val_pred = decode(val_scores, ship_thresholds, labels)
print(f"val macro-F1 (shipping thresholds, optimistic): {macro_f1(val_pred, val_gold, labels):.4f}")

with open("val_predictions.jsonl", "w", encoding="utf-8") as f:
    for row, pred in zip(val, val_pred):
        f.write(json.dumps({"id": row["id"], "labels": sorted(pred)}) + "\n")

test_pred = decode(test_scores, ship_thresholds, labels)
with open("predictions.jsonl", "w", encoding="utf-8") as f:
    for row, pred in zip(test, test_pred):
        f.write(json.dumps({"id": row["id"], "labels": sorted(pred)}) + "\n")

empty = sum(1 for p in test_pred if not p)
print(f"wrote {len(test)} predictions, {empty} ({100*empty/len(test):.0f}%) predicted no emotion")

val macro-F1 (shipping thresholds, optimistic): 0.4685
wrote 13263 predictions, 3910 (29%) predicted no emotion
